# Qwen3-VL Inference: HuggingFace vs nano-vllm

This tutorial compares two ways of running **Qwen3-VL-2B-Instruct** (a vision-language model):

1. **HuggingFace Transformers** -- the standard, easy-to-use reference implementation
2. **nano-vllm** -- a lightweight vLLM-style engine with paged KV cache, continuous batching, Flash Attention, and CUDA graphs

We will:
- Run identical prompts through both backends
- Profile prefill and decode latency
- Measure GPU memory usage
- Walk through *why* nano-vllm is faster and what each optimization does

**Prerequisites:** A machine with a GPU (>=16 GB VRAM), `transformers>=4.51`, `flash-attn`, `qwen-vl-utils`, and nano-vllm installed.

---
## Part 0: Setup

In [ ]:
import os, sys, time, gc
import torch
import numpy as np
from PIL import Image
from io import BytesIO
import requests

# Ensure nano-vllm is importable
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

torch.backends.cudnn.benchmark = True
print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  GPU: {torch.cuda.get_device_name()}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# --- Configuration ---
# Point this to your local copy of the model weights.
# If you don't have a local copy, use the HuggingFace model ID and it will auto-download.
MODEL_PATH = os.path.expanduser("~/huggingface/Qwen3-VL-2B-Instruct/")
if not os.path.isdir(MODEL_PATH):
    MODEL_PATH = "Qwen/Qwen3-VL-2B-Instruct"  # fallback to HF hub

IMAGE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
MAX_NEW_TOKENS = 256

print(f"Model: {MODEL_PATH}")

In [ ]:
# Download the test image once
response = requests.get(IMAGE_URL)
test_image = Image.open(BytesIO(response.content)).convert("RGB")
print(f"Image size: {test_image.size}")
test_image.resize((320, 240))

We'll use the same chat message format for both backends:

In [ ]:
# Shared prompt -- identical input to both backends
MESSAGES = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": IMAGE_URL},
            {"type": "text", "text": "Describe this image in detail. What objects do you see, and what is happening?"},
        ],
    }
]

print("Prompt ready.")

---
## Part 1: HuggingFace Transformers Baseline

The standard way to run Qwen3-VL. HuggingFace's `generate()` method handles everything:
tokenization, vision encoding, KV caching, and autoregressive decoding.

**What happens under the hood:**
- The processor tokenizes text and preprocesses images (resize, normalize, patchify)
- `model.generate()` runs the full forward pass eagerly for every token
- KV cache is allocated per-sequence as a dense tensor (no paging)
- Each decode step re-enters Python, selects from logits, and loops

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# Load model + processor
hf_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
hf_model.eval()
hf_processor = AutoProcessor.from_pretrained(MODEL_PATH)

print(f"HF model loaded. Parameters: {sum(p.numel() for p in hf_model.parameters()) / 1e9:.2f}B")
print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### 1.1 Preprocessing

The processor converts the chat messages into model inputs: `input_ids`, `attention_mask`,
`pixel_values`, and `image_grid_thw`.

In [ ]:
from qwen_vl_utils import process_vision_info

# Step 1: Apply chat template to get text with special tokens
text = hf_processor.apply_chat_template(MESSAGES, tokenize=False, add_generation_prompt=True)
print("Templated text (first 200 chars):")
print(text[:200], "...")

# Step 2: Extract images using Qwen's utility
image_inputs, video_inputs = process_vision_info(MESSAGES)
print(f"\nExtracted {len(image_inputs)} image(s)")

# Step 3: Processor tokenizes text + preprocesses images
hf_inputs = hf_processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    return_tensors="pt",
).to(hf_model.device)

print(f"\ninput_ids shape:      {hf_inputs['input_ids'].shape}")
print(f"pixel_values shape:   {hf_inputs['pixel_values'].shape}")
print(f"image_grid_thw:       {hf_inputs['image_grid_thw'].tolist()}")

num_prompt_tokens = hf_inputs["input_ids"].shape[1]
print(f"\nTotal prompt tokens: {num_prompt_tokens}")

### 1.2 Generation + Timing

We measure two phases separately:
- **Prefill**: process the entire prompt (vision encoder + first decoder pass). Compute-bound.
- **Decode**: generate tokens one by one. Memory-bandwidth-bound.

In [ ]:
def benchmark_hf(model, inputs, num_prompt_tokens, max_new_tokens, warmup=2, repeats=3):
    """Benchmark HuggingFace generate() with prefill/decode breakdown."""
    # Warmup
    for _ in range(warmup):
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=1)
    torch.cuda.synchronize()

    prefill_times = []
    total_times = []
    generated_tokens_list = []

    for _ in range(repeats):
        # Measure prefill (generate exactly 1 token)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out_1 = model.generate(**inputs, max_new_tokens=1)
        torch.cuda.synchronize()
        prefill_times.append(time.perf_counter() - t0)

        # Measure full generation (prefill + decode)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out_full = model.generate(**inputs, max_new_tokens=max_new_tokens)
        torch.cuda.synchronize()
        total_times.append(time.perf_counter() - t0)
        generated_tokens_list.append(out_full.shape[1] - num_prompt_tokens)

    prefill_time = np.median(prefill_times)
    total_time = np.median(total_times)
    num_generated = int(np.median(generated_tokens_list))
    decode_time = total_time - prefill_time

    return {
        "prefill_time_ms": prefill_time * 1000,
        "decode_time_ms": decode_time * 1000,
        "total_time_ms": total_time * 1000,
        "num_prompt_tokens": num_prompt_tokens,
        "num_generated_tokens": num_generated,
        "prefill_tok_per_s": num_prompt_tokens / prefill_time,
        "decode_tok_per_s": num_generated / decode_time if decode_time > 0 else 0,
        "output": out_full,
    }

In [ ]:
hf_results = benchmark_hf(hf_model, hf_inputs, num_prompt_tokens, MAX_NEW_TOKENS)

print("=" * 60)
print("HuggingFace Transformers Results")
print("=" * 60)
print(f"Prefill  ({hf_results['num_prompt_tokens']} tokens):")
print(f"  Time:       {hf_results['prefill_time_ms']:.1f} ms")
print(f"  Throughput: {hf_results['prefill_tok_per_s']:.0f} tok/s")
print(f"\nDecode   ({hf_results['num_generated_tokens']} tokens):")
print(f"  Time:       {hf_results['decode_time_ms']:.1f} ms")
print(f"  Throughput: {hf_results['decode_tok_per_s']:.1f} tok/s")
print(f"\nTotal:     {hf_results['total_time_ms']:.1f} ms")
print(f"GPU peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Decode the generated text
hf_output_ids = hf_results["output"][0][num_prompt_tokens:]
hf_text = hf_processor.decode(hf_output_ids, skip_special_tokens=True)
print("HF output:")
print(hf_text)

### 1.3 Understanding the Bottlenecks

HuggingFace's implementation is clean and correct, but it has several inefficiencies:

| Bottleneck | What happens | Impact |
|-----------|-------------|--------|
| **Eager execution** | Every decode step runs the full Python forward pass | High CPU overhead per token |
| **Dense KV cache** | Pre-allocates `[batch, max_len, heads, dim]` per sequence | Wastes memory, limits batch size |
| **No continuous batching** | Must wait for all sequences to finish | Underutilizes GPU on mixed-length outputs |
| **Python loop overhead** | Each token goes through Python sampling code | Adds ~1ms per token |

nano-vllm addresses all of these. Let's see how.

In [ ]:
# Save HF results before cleanup
hf_results_saved = {
    "prefill_time_ms": hf_results["prefill_time_ms"],
    "decode_time_ms": hf_results["decode_time_ms"],
    "total_time_ms": hf_results["total_time_ms"],
    "prefill_tok_per_s": hf_results["prefill_tok_per_s"],
    "decode_tok_per_s": hf_results["decode_tok_per_s"],
    "num_generated_tokens": hf_results["num_generated_tokens"],
}
hf_peak_memory = torch.cuda.max_memory_allocated() / 1e9

# Free HF model memory before loading nano-vllm
del hf_model, hf_inputs, hf_results
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"HF results saved. GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## Part 2: nano-vllm Engine

nano-vllm is a lightweight inference engine (~1,200 lines of Python) that implements the
core optimizations from [vLLM](https://github.com/vllm-project/vllm):

1. **Paged KV Cache** -- allocates memory in 256-token blocks, like virtual memory pages
2. **Continuous Batching** -- mixes prefill and decode steps, starts new requests as old ones finish
3. **Flash Attention** -- fused attention kernels from `flash-attn` (varlen for prefill, kvcache for decode)
4. **CUDA Graphs** -- captures the decode computation graph once, replays with zero CPU overhead
5. **Prefix Caching** -- reuses KV cache blocks across requests with matching prefixes (via xxhash)

For Qwen3-VL specifically, nano-vllm adds:
- **Vision encoder** with Flash Attention (varlen) and 2D rotary embeddings
- **M-RoPE** (Multimodal Rotary Position Embedding) for 3D positions `[temporal, height, width]`
- **DeepStack** injection of intermediate vision features into decoder layers
- **Spatial merge** to reduce visual tokens by 4x before feeding to the LLM

In [ ]:
from nanovllm import LLM, SamplingParams

# nano-vllm auto-detects VL models from the config
llm = LLM(
    MODEL_PATH,
    enforce_eager=True,      # Start without CUDA graphs (we'll enable them later)
    tensor_parallel_size=1,
)

print("nano-vllm engine loaded.")
print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### 2.1 How nano-vllm Handles VL Inference

Let's trace what happens when you call `llm.generate()` with an image:

```
User: messages + image
  |
  v
LLMEngine.add_request()
  |-- AutoProcessor: tokenize text + preprocess image
  |-- Creates Sequence(token_ids, pixel_values, image_grid_thw)
  |
  v
Scheduler.schedule() --> prefill batch
  |
  v
ModelRunner.prepare_prefill()
  |-- Compute 3D M-RoPE positions: text=[pos,pos,pos], image=[t,h,w]
  |-- Collect pixel_values across batch
  |-- Build cu_seqlens, slot_mapping for Flash Attention
  |
  v
ModelRunner.run_model(input_ids, positions_3d, pixel_values, grid_thw)
  |-- Vision encoder: Conv3D patch embed --> 24-layer ViT --> spatial merge
  |-- Scatter: replace <|image_pad|> tokens with visual embeddings
  |-- Decoder layers: M-RoPE attention + DeepStack injection
  |-- Sample first token
  |
  v
Scheduler.schedule() --> decode batch (no more vision data)
  |
  v
ModelRunner.run_model(last_token, position_1d)
  |-- Flash Attention with KV cache (no vision encoder!)
  |-- CUDA graph replay (if enabled)
  |-- Sample next token
  |
  v
Repeat decode until EOS or max_tokens
```

**Key insight:** The vision encoder only runs once during prefill. All subsequent decode
steps are pure text -- the visual information lives in the KV cache.

In [ ]:
# Run inference with nano-vllm
sampling_params = SamplingParams(temperature=0.6, max_tokens=MAX_NEW_TOKENS)
image_inputs, _ = process_vision_info(MESSAGES)

outputs = llm.generate(
    prompts=[MESSAGES],
    sampling_params=sampling_params,
    images=[image_inputs],
)

nano_text = outputs[0]["text"]
print("nano-vllm output:")
print(nano_text)

### 2.2 Benchmarking nano-vllm

nano-vllm's `generate()` already reports throughput via tqdm. But let's measure more precisely.

In [ ]:
def benchmark_nanovllm(llm, messages, image_inputs, sampling_params, warmup=2, repeats=3):
    """Benchmark nano-vllm with prefill/decode breakdown."""
    # Warmup
    for _ in range(warmup):
        llm.generate([messages], SamplingParams(temperature=0.6, max_tokens=1), images=[image_inputs], use_tqdm=False)

    prefill_times = []
    total_times = []
    generated_tokens_list = []

    for _ in range(repeats):
        # Prefill only (1 token)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out_1 = llm.generate([messages], SamplingParams(temperature=0.6, max_tokens=1), images=[image_inputs], use_tqdm=False)
        torch.cuda.synchronize()
        prefill_times.append(time.perf_counter() - t0)

        # Full generation
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out_full = llm.generate([messages], sampling_params, images=[image_inputs], use_tqdm=False)
        torch.cuda.synchronize()
        total_times.append(time.perf_counter() - t0)
        generated_tokens_list.append(len(out_full[0]["token_ids"]))

    prefill_time = np.median(prefill_times)
    total_time = np.median(total_times)
    num_generated = int(np.median(generated_tokens_list))
    decode_time = total_time - prefill_time

    return {
        "prefill_time_ms": prefill_time * 1000,
        "decode_time_ms": decode_time * 1000,
        "total_time_ms": total_time * 1000,
        "num_generated_tokens": num_generated,
        "prefill_tok_per_s": num_prompt_tokens / prefill_time,  # uses same prompt token count
        "decode_tok_per_s": num_generated / decode_time if decode_time > 0 else 0,
    }

In [ ]:
nano_results_eager = benchmark_nanovllm(llm, MESSAGES, image_inputs, sampling_params)

print("=" * 60)
print("nano-vllm Results (enforce_eager=True, no CUDA graphs)")
print("=" * 60)
print(f"Prefill  ({num_prompt_tokens} tokens):")
print(f"  Time:       {nano_results_eager['prefill_time_ms']:.1f} ms")
print(f"  Throughput: {nano_results_eager['prefill_tok_per_s']:.0f} tok/s")
print(f"\nDecode   ({nano_results_eager['num_generated_tokens']} tokens):")
print(f"  Time:       {nano_results_eager['decode_time_ms']:.1f} ms")
print(f"  Throughput: {nano_results_eager['decode_tok_per_s']:.1f} tok/s")
print(f"\nTotal:     {nano_results_eager['total_time_ms']:.1f} ms")
print(f"GPU peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Clean up eager engine
llm.exit()
del llm
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

### 2.3 Enabling CUDA Graphs

CUDA graphs capture the entire GPU computation for a decode step (all 28 decoder layers)
into a single replayable graph. This eliminates per-token CPU overhead.

**How it works in nano-vllm:**
1. During init, graphs are captured for batch sizes `[1, 2, 4, 8, 16, 32, ...]`
2. At decode time, the engine picks the smallest graph that fits the current batch
3. Input tensors are copied into pre-allocated buffers, the graph replays, and outputs are read back
4. **Zero Python overhead** per decode step -- just a single `graph.replay()` call

Note: CUDA graphs only apply to the decode phase. Prefill runs eagerly because
sequence lengths vary (and the vision encoder is only needed once).

In [ ]:
# Reload with CUDA graphs enabled
llm_graphs = LLM(
    MODEL_PATH,
    enforce_eager=False,     # Enable CUDA graphs for decode
    tensor_parallel_size=1,
)

print("nano-vllm with CUDA graphs loaded.")

In [ ]:
nano_results_graphs = benchmark_nanovllm(llm_graphs, MESSAGES, image_inputs, sampling_params)

print("=" * 60)
print("nano-vllm Results (CUDA graphs enabled)")
print("=" * 60)
print(f"Prefill  ({num_prompt_tokens} tokens):")
print(f"  Time:       {nano_results_graphs['prefill_time_ms']:.1f} ms")
print(f"  Throughput: {nano_results_graphs['prefill_tok_per_s']:.0f} tok/s")
print(f"\nDecode   ({nano_results_graphs['num_generated_tokens']} tokens):")
print(f"  Time:       {nano_results_graphs['decode_time_ms']:.1f} ms")
print(f"  Throughput: {nano_results_graphs['decode_tok_per_s']:.1f} tok/s")
print(f"\nTotal:     {nano_results_graphs['total_time_ms']:.1f} ms")
print(f"GPU peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

---
## Part 3: Head-to-Head Comparison

In [ ]:
print(f"{'Metric':<30} {'HF':>12} {'nano (eager)':>14} {'nano (graph)':>14}")
print("-" * 72)

rows = [
    ("Prefill time (ms)",
     hf_results_saved["prefill_time_ms"],
     nano_results_eager["prefill_time_ms"],
     nano_results_graphs["prefill_time_ms"]),
    ("Prefill throughput (tok/s)",
     hf_results_saved["prefill_tok_per_s"],
     nano_results_eager["prefill_tok_per_s"],
     nano_results_graphs["prefill_tok_per_s"]),
    ("Decode time (ms)",
     hf_results_saved["decode_time_ms"],
     nano_results_eager["decode_time_ms"],
     nano_results_graphs["decode_time_ms"]),
    ("Decode throughput (tok/s)",
     hf_results_saved["decode_tok_per_s"],
     nano_results_eager["decode_tok_per_s"],
     nano_results_graphs["decode_tok_per_s"]),
    ("Total time (ms)",
     hf_results_saved["total_time_ms"],
     nano_results_eager["total_time_ms"],
     nano_results_graphs["total_time_ms"]),
]

for name, hf_val, eager_val, graph_val in rows:
    print(f"{name:<30} {hf_val:>12.1f} {eager_val:>14.1f} {graph_val:>14.1f}")

If you ran all cells sequentially, you should see a clean comparison table above.
The HF results were saved before freeing GPU memory.

In [ ]:
# Calculate speedups
print("Speedup (nano-vllm with CUDA graphs vs HuggingFace):")
print(f"  Prefill: {hf_results_saved["prefill_time_ms"] / nano_results_graphs["prefill_time_ms"]:.2f}x")
print(f"  Decode:  {hf_results_saved["decode_time_ms"] / nano_results_graphs["decode_time_ms"]:.2f}x")
print(f"  Total:   {hf_results_saved["total_time_ms"] / nano_results_graphs["total_time_ms"]:.2f}x")

---
## Part 4: Deep Dive -- What Makes nano-vllm Faster?

Let's walk through each optimization and see the actual code.

### 4.1 Paged KV Cache

**Problem:** HF allocates a dense `[batch, max_len, heads, dim]` KV cache per sequence.
If `max_len = 4096` but the actual sequence is only 200 tokens, 95% of the memory is wasted.

**Solution:** nano-vllm divides KV cache into fixed-size **blocks** (default 256 tokens).
Blocks are allocated on demand, like virtual memory pages.

```
KV Cache Pool (one big tensor):
┌────────┬────────┬────────┬────────┬────────┬────────┐
│ Block 0│ Block 1│ Block 2│ Block 3│ Block 4│  ...   │
└────────┴────────┴────────┴────────┴────────┴────────┘

Sequence A (300 tokens):  block_table = [3, 0]    (2 blocks)
Sequence B (100 tokens):  block_table = [5]       (1 block)
Sequence C (500 tokens):  block_table = [1, 4, 2] (3 blocks)
```

Each sequence has a `block_table` mapping logical block indices to physical block IDs.
Flash Attention reads from the correct blocks using this table.

In [ ]:
# Let's look at the actual KV cache allocation
from nanovllm.engine.model_runner import ModelRunner
import inspect

# Show the allocate_kv_cache method
source = inspect.getsource(ModelRunner.allocate_kv_cache)
print(source)

The KV cache is a single pre-allocated tensor:
```python
# Shape: [2, num_layers, num_blocks, block_size, num_kv_heads, head_dim]
# 2 = one for K, one for V
self.kv_cache = torch.empty(2, num_layers, num_kvcache_blocks, block_size, num_kv_heads, head_dim)
```

The number of blocks is computed to fill available GPU memory:
```python
num_kvcache_blocks = (total_gpu * utilization - model_weights - peak_activation) / bytes_per_block
```

### 4.2 Flash Attention (Prefill vs Decode)

nano-vllm uses two different Flash Attention kernels depending on the phase:

| Phase | Kernel | Why |
|-------|--------|-----|
| **Prefill** | `flash_attn_varlen_func` | Processes variable-length sequences in a single batch. Uses `cu_seqlens` to mark boundaries. |
| **Decode** | `flash_attn_with_kvcache` | Single-query attention against the full KV cache. Reads from block-table-indexed cache. |

Additionally, a custom **Triton kernel** stores new KV pairs into the paged cache:

In [ ]:
from nanovllm.layers.attention import Attention
source = inspect.getsource(Attention.forward)
print(source)

The Triton `store_kvcache_kernel` writes K and V vectors to their correct slots
in the block-based cache. Each token's slot is computed as:
```
slot = block_table[block_idx] * block_size + offset_within_block
```

### 4.3 Continuous Batching

HuggingFace processes one batch at a time and waits for all sequences to finish.
nano-vllm's scheduler interleaves prefill and decode:

```
Step 1: [prefill A, prefill B]         -- both start
Step 2: [decode A, decode B]           -- both generate 1 token
Step 3: [decode A, decode B]           -- A finishes (hits EOS)
Step 4: [prefill C, decode B]          -- C starts immediately, B continues
Step 5: [decode C, decode B]           -- both generating
```

This keeps the GPU busy and reduces latency for short outputs.

In [ ]:
from nanovllm.engine.scheduler import Scheduler
source = inspect.getsource(Scheduler.schedule)
print(source)

The scheduler has a `waiting` queue (new requests) and a `running` queue (in-progress decodes).
It prioritizes prefill when there are waiting requests, otherwise batches all running decodes together.

Note the **preemption** logic: if there aren't enough KV cache blocks for a decode step,
the scheduler evicts the most recent sequence back to `waiting` and frees its blocks.

### 4.4 CUDA Graphs for Decode

During decode, every step runs the same computation (28 decoder layers) with different inputs.
CUDA graphs capture this computation once and replay it:

```python
# Capture (happens once during init)
with torch.cuda.graph(graph):
    outputs = model(input_ids, positions)  # record GPU ops

# Replay (happens every decode step -- near-zero CPU overhead)
graph_vars["input_ids"][:bs] = new_input_ids  # write to pre-allocated buffers
graph_vars["positions"][:bs] = new_positions
graph.replay()                                  # single GPU call
result = graph_vars["outputs"][:bs]             # read from pre-allocated buffers
```

Graphs are pre-captured for batch sizes `[1, 2, 4, 8, 16, 32, ...]` up to `max_num_seqs`.
At runtime, the smallest graph >= current batch size is selected.

### 4.5 Prefix Caching

When multiple requests share the same prefix (common in chat/system prompts),
nano-vllm reuses KV cache blocks:

```
Request 1: "You are a helpful assistant. User: What is 2+2?"
Request 2: "You are a helpful assistant. User: Explain gravity."
          └────────── shared prefix ──────────┘
```

Each block is hashed (via xxhash) by its token content + parent block hash.
When a new request's block hash matches an existing block, that block is reused
instead of re-computing its KV pairs.

In [ ]:
from nanovllm.engine.block_manager import BlockManager
source = inspect.getsource(BlockManager.allocate)
print(source)

---
## Part 5: VL-Specific Optimizations

Beyond the general inference optimizations, nano-vllm handles the vision-language
pipeline efficiently.

### 5.1 M-RoPE: Multimodal Rotary Position Embeddings

Standard RoPE assigns each token a single position `p` and computes:
```
cos(p * freq), sin(p * freq)  for each frequency in the head dimension
```

M-RoPE extends this to **3 axes** for vision tokens:
- **Temporal** (t): video frame index
- **Height** (h): vertical position in the image grid
- **Width** (w): horizontal position in the image grid

The head dimension is split into 3 sections `[24, 20, 20]` (for Qwen3-VL-2B):
```
head_dim = 128 → half = 64
Section 0 (temporal): 24 frequency pairs → uses t positions
Section 1 (height):   20 frequency pairs → uses h positions
Section 2 (width):    20 frequency pairs → uses w positions
```

For text tokens, all 3 position axes are identical: `[pos, pos, pos]`.
For image tokens, they differ: `[t_idx, h_idx, w_idx]`.

In [ ]:
from nanovllm.layers.rotary_embedding import MRoPEEmbedding

# Show the 3D forward path
source = inspect.getsource(MRoPEEmbedding.forward_3d)
print(source)

Notice how each axis looks up its own section of the frequency table:
```python
cos_t = self.cos_cache[positions[0]][:, :s0]        # temporal positions → first 24 freqs
cos_h = self.cos_cache[positions[1]][:, s0:s0+s1]   # height positions  → next 20 freqs
cos_w = self.cos_cache[positions[2]][:, s0+s1:]      # width positions   → last 20 freqs
```

The **interleave permutation** reshuffles from `[TTT...HHH...WWW...]` to `[THWTHW...]`
for compute efficiency.

### 5.2 Position Computation for VL Sequences

Let's trace how 3D positions are computed for a sequence with text and images:

In [ ]:
source = inspect.getsource(ModelRunner._compute_mrope_positions)
print(source)

In [ ]:
# Visualize position assignment for a toy example
# Suppose: [text, text, image(2x3 grid), text, text]
# image_token_id = 151655, spatial_merge_size = 2

print("Example: 'Hello <image_2x3_grid> world!'")
print()
print("Token:    [Hello] [,]  [img] [img] [img] [img] [img] [img]  [world] [!]")
print("                       └─────── 2x3 merged grid ──────┘")
print()
print("t_pos:    [  0  ] [ 1]  [ 2 ] [ 2 ] [ 2 ] [ 2 ] [ 2 ] [ 2 ]  [  5  ] [ 6]")
print("h_pos:    [  0  ] [ 1]  [ 2 ] [ 2 ] [ 3 ] [ 3 ] [ 4 ] [ 4 ]  [  5  ] [ 6]")
print("w_pos:    [  0  ] [ 1]  [ 2 ] [ 3 ] [ 2 ] [ 3 ] [ 2 ] [ 3 ]  [  5  ] [ 6]")
print()
print("Text tokens: all 3 axes have the SAME position (= standard 1D RoPE)")
print("Image tokens: t=temporal_idx, h=row_idx, w=col_idx in the merged grid")
print("After the image, text offset jumps by max(t=1, h=3, w=2) = 3")
print("So 'world' starts at position 5 (= 2 text tokens + 3 from image extent)")

### 5.3 Vision Encoder + Scatter + DeepStack

The vision pipeline has three stages:

**1. Vision Encoder** (24-layer ViT with Flash Attention):
```
pixel_values → Conv3D patchify → 24x [LayerNorm → FlashAttn(2D RoPE) → LayerNorm → MLP]
                                                                                    ↓
                                                              Spatial merge (2x2 → 1 token)
                                                                                    ↓
                                                              visual_embeds [N_visual, 2048]
```

**2. Scatter** replaces placeholder tokens with visual embeddings:
```python
image_mask = (input_ids == image_token_id)         # find <|image_pad|> positions
hidden_states[image_mask] = visual_embeds           # scatter visual features in
```

**3. DeepStack** injects intermediate ViT features into decoder layers:
```python
# At decoder layer i, if it's a DeepStack injection point:
hidden_states[image_mask] += deepstack_features[ds_idx]
```

In [ ]:
from nanovllm.models.qwen3_vl import Qwen3VLModel
source = inspect.getsource(Qwen3VLModel.forward)
print(source)

### 5.4 Vision Encoder 2D RoPE (separate from text M-RoPE)

The vision encoder has its own rotary embeddings, computed from the spatial grid:

In [ ]:
from nanovllm.models.qwen3_vl import compute_vision_rotary_emb, Qwen3VLPatchMerger

print("--- Vision 2D RoPE computation ---")
source = inspect.getsource(compute_vision_rotary_emb)
print(source)

print("\n--- Spatial Merge (2x2 patches -> 1 token) ---")
source = inspect.getsource(Qwen3VLPatchMerger.forward)
print(source)

### 5.5 Decode Phase: Vision is Free

A critical optimization: **the vision encoder only runs during prefill**.

After prefill:
- Visual information is encoded in the KV cache
- The `pixel_values` tensor is freed (`seq.pixel_values = None`)
- Decode steps are pure text -- identical to a text-only model
- CUDA graphs can be used (the decode graph is the same as for text-only Qwen3)

This means nano-vllm's decode speed for a VL model should be similar to its
decode speed for a same-sized text-only model.

---
## Part 6: Batch Throughput Comparison

The real power of continuous batching shows when you have **multiple requests**.
Let's compare throughput on a batch of prompts.

In [ ]:
# Create a batch of different prompts with the same image
QUESTIONS = [
    "Describe this image in detail.",
    "What colors are prominent in this image?",
    "How many people or animals are visible?",
    "What is the setting or location in this image?",
]

batch_messages = []
batch_images = []
for q in QUESTIONS:
    msg = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": IMAGE_URL},
                {"type": "text", "text": q},
            ],
        }
    ]
    batch_messages.append(msg)
    img_inputs, _ = process_vision_info(msg)
    batch_images.append(img_inputs)

print(f"Batch of {len(QUESTIONS)} prompts ready.")

In [ ]:
# Benchmark batch generation with nano-vllm
sampling_params_batch = SamplingParams(temperature=0.6, max_tokens=128)

torch.cuda.synchronize()
t0 = time.perf_counter()
batch_outputs = llm_graphs.generate(
    prompts=batch_messages,
    sampling_params=sampling_params_batch,
    images=batch_images,
    use_tqdm=True,
)
torch.cuda.synchronize()
batch_time = time.perf_counter() - t0

total_tokens = sum(len(o["token_ids"]) for o in batch_outputs)
print(f"\nBatch of {len(QUESTIONS)} prompts:")
print(f"  Total time:    {batch_time*1000:.0f} ms")
print(f"  Total tokens:  {total_tokens}")
print(f"  Throughput:    {total_tokens / batch_time:.1f} tok/s")
print(f"  Per-request:   {batch_time / len(QUESTIONS) * 1000:.0f} ms avg")

In [ ]:
# Show all outputs
for q, out in zip(QUESTIONS, batch_outputs):
    print(f"\nQ: {q}")
    print(f"A: {out['text'][:200]}{'...' if len(out['text']) > 200 else ''}")

With continuous batching, all 4 requests are served concurrently:
- Prefill happens sequentially (one at a time, since each prefill is large)
- Decode happens in parallel (all 4 sequences generate tokens together)
- When one sequence finishes early, its KV cache blocks are freed immediately

The total throughput should be higher than 4x the single-request throughput
because decode is memory-bandwidth-bound and batching amortizes the cost.

---
## Part 7: Memory Analysis

Let's compare how the two backends use GPU memory.

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(MODEL_PATH)
text_config = config.text_config
vision_config = config.vision_config

print("=== Qwen3-VL-2B Memory Breakdown ===")
print()

# Model weights
# Vision encoder
vis_hidden = vision_config.hidden_size       # 1024
vis_layers = vision_config.depth             # 24
vis_inter = vision_config.intermediate_size  # 4096

vis_params = (
    3 * 16 * vis_hidden * vis_hidden    # patch_embed Conv3d (approx)
    + vis_layers * (3 * vis_hidden * vis_hidden  # QKV
                    + vis_hidden * vis_hidden      # output proj
                    + 2 * vis_hidden * vis_inter    # MLP
                    + 4 * vis_hidden)               # LayerNorms
)

# Text decoder
txt_hidden = text_config.hidden_size          # 2048
txt_layers = text_config.num_hidden_layers    # 28
txt_inter = text_config.intermediate_size     # 8192
vocab = text_config.vocab_size                # 151936

txt_params = (
    vocab * txt_hidden                         # embed_tokens
    + txt_layers * (3 * txt_hidden * txt_hidden  # QKV (but GQA: actually Q=2048*2048, KV=2048*512)
                    + txt_hidden * txt_hidden      # output proj
                    + 3 * txt_hidden * txt_inter    # gate, up, down
                    + 4 * txt_hidden)               # RMSNorms
)

bytes_per_param = 2  # bf16
vis_gb = vis_params * bytes_per_param / 1e9
txt_gb = txt_params * bytes_per_param / 1e9

print(f"Vision encoder (approx): {vis_gb:.2f} GB")
print(f"Text decoder (approx):   {txt_gb:.2f} GB")
print(f"Total weights (approx):  {vis_gb + txt_gb:.2f} GB")

# KV cache per token
num_kv_heads = text_config.num_key_value_heads   # 8
head_dim = text_config.hidden_size // text_config.num_attention_heads  # 128
kv_per_token = 2 * txt_layers * num_kv_heads * head_dim * bytes_per_param
print(f"\nKV cache per token: {kv_per_token} bytes ({kv_per_token/1024:.1f} KB)")
print(f"KV cache for 1K tokens: {kv_per_token * 1024 / 1e6:.1f} MB")
print(f"KV cache for 4K tokens: {kv_per_token * 4096 / 1e6:.1f} MB")

# nano-vllm block size
block_bytes = 2 * txt_layers * 256 * num_kv_heads * head_dim * bytes_per_param
print(f"\nnano-vllm KV block (256 tokens): {block_bytes / 1e6:.1f} MB")

### HF vs nano-vllm Memory Strategy

| | HuggingFace | nano-vllm |
|---|---|---|
| **Allocation** | Dense: `[batch, max_len, heads, dim]` per sequence | Paged: 256-token blocks, on demand |
| **Waste** | Unused positions are allocated but empty | Only allocated blocks have memory |
| **Sharing** | No sharing between requests | Prefix caching shares blocks with identical content |
| **Eviction** | Not supported | Preemption: evict sequences when blocks run out |
| **Max concurrent** | Limited by pre-allocated cache | Can serve more sequences with less memory |

---
## Part 8: Architecture Summary

Here's the complete picture of what makes nano-vllm different from HuggingFace for VL inference:

```
┌─────────────────────────────────────────────────────────────────┐
│                        nano-vllm Engine                        │
│                                                                 │
│  ┌─────────────┐     ┌─────────────┐     ┌─────────────────┐  │
│  │  LLMEngine  │────>│  Scheduler   │────>│  ModelRunner    │  │
│  │             │     │              │     │                 │  │
│  │ - processor │     │ - waiting Q  │     │ - prepare_      │  │
│  │ - tokenizer │     │ - running Q  │     │   prefill()     │  │
│  │ - add_      │     │ - preemption │     │ - prepare_      │  │
│  │   request() │     │              │     │   decode()      │  │
│  └─────────────┘     └──────┬───────┘     │ - run_model()   │  │
│                             │             │ - CUDA graphs   │  │
│                    ┌────────┴────────┐    └────────┬────────┘  │
│                    │  BlockManager   │             │            │
│                    │                 │             │            │
│                    │ - paged blocks  │    ┌────────┴────────┐  │
│                    │ - prefix cache  │    │  Qwen3VL Model  │  │
│                    │ - xxhash        │    │                 │  │
│                    └─────────────────┘    │ ┌─────────────┐ │  │
│                                           │ │VisionEncoder│ │  │
│                                           │ │ (prefill)   │ │  │
│          ┌───────────────────────┐        │ └──────┬──────┘ │  │
│          │     KV Cache Pool     │        │        │scatter │  │
│          │  [2, layers, blocks,  │<───────│ ┌──────┴──────┐ │  │
│          │   block_size,         │        │ │  Decoder    │ │  │
│          │   kv_heads, head_dim] │        │ │  + M-RoPE   │ │  │
│          └───────────────────────┘        │ │  + DeepStack│ │  │
│                                           │ └─────────────┘ │  │
│                                           └─────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

---
## Part 9: Exercises

1. **Try a different image**: Replace `IMAGE_URL` with your own image URL and re-run the comparison.

2. **Vary `max_tokens`**: How does decode throughput change as you generate 32, 128, 512, 1024 tokens?
   The decode phase should show a near-linear increase in time since each step takes the same amount.

3. **Batch size scaling**: Create batches of 1, 2, 4, 8 requests and plot total throughput (tok/s).
   You should see throughput increase due to better GPU utilization.

4. **Read the source**: Follow these files to understand the full pipeline:
   - `nanovllm/models/qwen3_vl.py` -- vision encoder + scatter + DeepStack
   - `nanovllm/layers/rotary_embedding.py` -- M-RoPE implementation
   - `nanovllm/engine/model_runner.py` -- prefill/decode orchestration
   - `nanovllm/layers/attention.py` -- Flash Attention + Triton KV cache store

5. **Profile with `torch.profiler`**: Wrap a single `llm.generate()` call in a profiler to see
   the GPU kernel timeline. Compare the number of kernel launches between HF and nano-vllm.

6. **Compare vision token counts**: Change the input image resolution and observe how
   `image_grid_thw` changes. A larger image produces more visual tokens after spatial merge,
   which increases prefill time but not decode time.

---
## Summary

| Feature | HuggingFace | nano-vllm |
|---------|------------|----------|
| **Ease of use** | Very high -- 5 lines of code | Moderate -- needs model path, understanding of VL API |
| **Prefill speed** | Good (Flash Attention available) | Better (fused varlen + Triton KV store) |
| **Decode speed** | Moderate (Python loop overhead) | Fast (CUDA graphs, no Python overhead) |
| **Memory efficiency** | Dense KV cache | Paged KV cache (block-based, prefix caching) |
| **Batching** | Static (wait for all to finish) | Continuous (start new requests immediately) |
| **Throughput scaling** | Linear with batch size | Better than linear (amortized overhead) |
| **Code complexity** | Abstracted away | ~1,200 lines of readable Python |

**For learning:** Start with HuggingFace to understand the model. Then read nano-vllm's source
to understand how production inference engines optimize every step.

**For production:** Use an inference engine (vLLM, nano-vllm, TensorRT-LLM) for the speed
and memory benefits. The API is slightly more complex, but the performance gains are significant.

In [ ]:
# Cleanup
llm_graphs.exit()
del llm_graphs
gc.collect()
torch.cuda.empty_cache()
print("Done! GPU memory released.")